# 01 — Baselines: la media y la regresión lineal

**Componente B — predicción de rinde.** El problema es de *regresión supervisada*:
estimar el rinde (`rinde_kgha`, kg/ha) de un departamento-campaña a partir de las
variables climáticas de la campaña.

Antes de tunear modelos hay que fijar el **piso**: ¿qué tan lejos llega lo trivial?
Este notebook establece tres referencias en orden creciente de sofisticación:

1. **Media global** — predecir siempre el rinde medio de train. El baseline más
   tonto posible; su R² es 0 por construcción sobre train.
2. **Media por departamento** (climatología) — predecir, para cada fila, el rinde
   histórico medio de su departamento. Captura la estructura *espacial* (un depto
   rinde sistemáticamente más que otro) y es el baseline agronómico honesto.
3. **Regresión lineal (OLS)** — el primer modelo que usa el clima.

Los datos, el split temporal (train ≤2020, test ≥2021) y las features están en
`datos.py`; las métricas y baselines en `evaluacion.py`.

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))          # componente_b/ (datos, evaluacion)
warnings.filterwarnings('ignore')                  # silenciar ConvergenceWarning de sklearn

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

import datos, evaluacion as ev
from modelos import (LinearRegressor, XGBoostRegressor, NeuralNetRegressor,
                     RandomForestRegressorModel, HistGBMRegressor, StackingRegressorModel)

# Cultivo del estudio (cambiar a 'maiz' para reproducir con maíz).
CULTIVO = 'soja'
ds = datos.prepare(CULTIVO, use_agro=True, enc_smooth=10.0)
print(f'{CULTIVO}: {len(ds.feature_cols)} features | '
      f'train {ds.X_train.shape[0]} filas (≤{datos.TRAIN_END}) | '
      f'test {ds.X_test.shape[0]} filas (≥{datos.TEST_START})')


## Los datos

Las features que ve el modelo son de tres tipos (todas escaladas con parámetros de
train, sin leakage): **clima mensual** (Sep–Mar), **codificación del departamento**
por su rinde medio en train (estructura espacial) y el **año** (tendencia
tecnológica). El target es el rinde crudo en kg/ha.

In [ ]:
print('features:', ds.feature_cols[:6], '...', ds.feature_cols[-2:])
print(f'\nrinde train: media={ds.y_train.mean():.0f}  std={ds.y_train.std():.0f} kg/ha')
print(f'rinde test : media={ds.y_test.mean():.0f}  std={ds.y_test.std():.0f} kg/ha')
ds.meta_test.head(3)

## Baselines

Para regresión reportamos cuatro métricas (definidas en `evaluacion.metricas`):
**MAE** y **RMSE** en kg/ha (RMSE castiga más los errores grandes), **R²** (varianza
explicada; 0 = tan bueno como predecir la media, negativo = peor) y **MAPE** (error
porcentual).

In [ ]:
filas = [
    ev.evaluar('media global',  ev.pred_media(ds),      ds),
    ev.evaluar('media x depto',  ev.pred_media_depto(ds), ds),
]
tabla_base = ev.tabla_comparativa(filas, ordenar_por='rmse')
tabla_base

La **media por departamento** ya explica bastante más varianza que la media global:
casi todo el poder predictivo "fácil" del rinde es *dónde* está el campo, no el
clima del año. Ese es el número que cualquier modelo con clima tiene que superar.

## Regresión lineal (OLS, sin regularizar)

Primer modelo con clima. Sin tuning todavía — eso es el notebook 02.

In [ ]:
ols = LinearRegressor(penalty='none').fit(ds.X_train, ds.y_train)
pred_ols = ols.predict(ds.X_test)

filas.append(ev.evaluar('OLS (lineal)', pred_ols, ds))
tabla = ev.tabla_comparativa(filas, ordenar_por='rmse')
tabla

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
ev.plot_pred_vs_real(ds.y_test, ev.pred_media_depto(ds), 'Baseline: media x depto',
                     color=ev.C_BASE, ax=axes[0])
ev.plot_pred_vs_real(ds.y_test, pred_ols, 'OLS (lineal)', color=ev.C_LINEAR, ax=axes[1])
plt.tight_layout(); plt.show()

## Conclusión

- La **media por departamento** es un baseline fuerte: el grueso de la señal del
  rinde es espacial.
- La **OLS** aprovecha el clima y la tendencia y mejora sobre la media por depto,
  pero sin regularizar es propensa a la colinealidad (las 54 columnas mensuales
  están muy correlacionadas entre sí).

Los notebooks siguientes tunean cada familia de modelos: regularización para el
lineal (02), XGBoost (03) y red neuronal (04); la comparación final es el 05.